# MHRQI: Monte Carlo Simulations & Interactive Demos

## Overview

This notebook demonstrates MHRQI's Monte Carlo simulation backend for efficient hierarchical quantum image processing. Unlike full statevector simulation, Monte Carlo sampling enables realistic noise characterization and performance evaluation on larger images.

**Key Concepts:**
- Shot-based sampling (Monte Carlo method)
- Convergence analysis
- Adaptive shot allocation
- GPU acceleration (if available)

---

## Setup

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from mhrqi import MHRQI
from mhrqi.utils import general as utils
from mhrqi.core.simulation import MonteCarloSimulator, HierarchicalMeasurementAggregator
from mhrqi.utils import monte_carlo as mc_utils

# Check GPU availability
mc_sim = MonteCarloSimulator(use_gpu=True)
print(f"GPU available: {mc_sim.gpu_available}")

# Set style for plotting
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Understanding Monte Carlo Sampling

Instead of computing the full quantum state ($2^n$ amplitudes), we sample measurement outcomes statistically.

In [ ]:
# Create a synthetic image
np.random.seed(42)
img_size = 16  # 16x16 for demonstration
img = np.random.rand(img_size, img_size) * 0.8  # Random intensities [0, 0.8]

# Add layer structure (simulating OCT-like appearance)
for row in range(img_size):
    intensity = 0.3 + 0.4 * (1 - abs(row - img_size//2) / (img_size//2))
    img[row, :] = np.minimum(intensity, 1.0)

# Add speckle noise
speckle = np.random.exponential(0.1, img.shape)
img_noisy = np.minimum(img + speckle, 1.0)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Clean Image')
axes[1].imshow(speckle, cmap='hot')
axes[1].set_title('Speckle Noise')
axes[2].imshow(img_noisy, cmap='gray')
axes[2].set_title('Noisy Image')
plt.tight_layout()
plt.show()

print(f"Image shape: {img_noisy.shape}")
print(f"Noise level (MSE): {np.mean((img_noisy - img)**2):.4f}")

## 2. MHRQI Encoding with Monte Carlo Sampling

In [ ]:
# Initialize MHRQI
depth = int(np.log2(img_size))  # log2(16) = 4
model = MHRQI(depth=depth, bit_depth=8)

print(f"Circuit Configuration:")
print(f"  Image size: {img_size}×{img_size}")
print(f"  Hierarchy depth: {depth}")
print(f"  Position qubits: {2*depth}")
print(f"  Intensity qubits: 8")
print(f"  Total qubits: {2*depth + 8 + 1 + 2}")

# Generate hierarchical coordinates
hc_matrix = utils.generate_hierarchical_coord_matrix(img_size, d=2)
print(f"\nHierarchical coordinate vectors: {len(hc_matrix)}")
print(f"Sample HCV for pixel (0,0): {hc_matrix[0]}")
print(f"Sample HCV for pixel (15,15): {hc_matrix[-1]}")

In [ ]:
# Option A: Fast lazy upload (for large circuits)
model.lazy_upload(hc_matrix, img_noisy)
print("✓ Image uploaded via lazy statevector initialization")

## 3. Monte Carlo Shot Convergence Analysis

In [ ]:
# Analyze convergence behavior
print("Analyzing Monte Carlo convergence...")
print("(This estimates how many shots are needed for accurate results)\n")

# Estimate shots needed for different target accuracies
target_errors = [0.10, 0.05, 0.01, 0.001]
shots_needed = {}

for error in target_errors:
    shots = mc_utils.estimate_required_shots(
        target_accuracy=0.95,
        target_error=error,
        num_outcomes=2**depth
    )
    shots_needed[error] = shots
    print(f"Target error {error:.3f} → {shots:,} shots needed")

print(f"\nFor this demo, we'll use 10,000 shots (good balance)")

## 4. Run Simulation with Monte Carlo Backend

In [ ]:
# Simulate WITHOUT denoising first
print("Simulating MHRQI without denoising (10,000 Monte Carlo shots)...")
result_no_denoise = model.simulate(shots=10000, monte_carlo_seed=42)
img_reconstructed = result_no_denoise.reconstruct(use_denoising_bias=False)

# Compute metrics
metrics_no_denoise = result_no_denoise.compute_metrics(reference_image=img)

print(f"\nReconstruction Metrics (without denoising):")
print(f"  MSE:  {metrics_no_denoise.get('mse', 0):.6f}")
print(f"  PSNR: {metrics_no_denoise.get('psnr', 0):.2f} dB")
print(f"  SSIM: {metrics_no_denoise.get('ssim', 0):.4f}")

## 5. Hierarchical Consistency Denoising

In [ ]:
# Reset model and apply denoising
model2 = MHRQI(depth=depth, bit_depth=8)
model2.lazy_upload(hc_matrix, img_noisy)
model2.apply_denoising()  # Apply quantum denoising circuit

print("Simulating MHRQI WITH hierarchical denoising (10,000 shots)...")
result_with_denoise = model2.simulate(shots=10000, monte_carlo_seed=42)
img_denoised = result_with_denoise.reconstruct(use_denoising_bias=True)

# Compute metrics
metrics_with_denoise = result_with_denoise.compute_metrics(reference_image=img)

print(f"\nReconstruction Metrics (with denoising):")
print(f"  MSE:  {metrics_with_denoise.get('mse', 0):.6f}")
print(f"  PSNR: {metrics_with_denoise.get('psnr', 0):.2f} dB")
print(f"  SSIM: {metrics_with_denoise.get('ssim', 0):.4f}")

## 6. Visual Comparison

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: Reference and noisy
axes[0, 0].imshow(img, cmap='gray')
axes[0, 0].set_title('Reference Image')
axes[0, 0].axis('off')

axes[0, 1].imshow(img_noisy, cmap='gray')
axes[0, 1].set_title(f'Noisy Input (MSE={np.mean((img_noisy-img)**2):.4f})')
axes[0, 1].axis('off')

axes[0, 2].imshow(np.abs(img_noisy - img), cmap='hot')
axes[0, 2].set_title('Noise Pattern')
axes[0, 2].axis('off')

# Row 2: Reconstructions
axes[1, 0].imshow(img_reconstructed, cmap='gray')
axes[1, 0].set_title(f'MHRQI (No Denoise)\nPSNR={metrics_no_denoise.get("psnr", 0):.2f} dB')
axes[1, 0].axis('off')

axes[1, 1].imshow(img_denoised, cmap='gray')
axes[1, 1].set_title(f'MHRQI + Denoising\nPSNR={metrics_with_denoise.get("psnr", 0):.2f} dB')
axes[1, 1].axis('off')

# Difference maps
diff_denoise = np.abs(img_denoised - img)
axes[1, 2].imshow(diff_denoise, cmap='hot')
axes[1, 2].set_title(f'Residual Error\nMSE={np.mean(diff_denoise**2):.4f}')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## 7. Multi-Shot Comparison

In [ ]:
# Test different shot counts
shot_counts = [100, 500, 1000, 5000, 10000]
psnr_values = []

print("Running convergence analysis with different shot counts...")
for shots in shot_counts:
    model_test = MHRQI(depth=depth, bit_depth=8)
    model_test.lazy_upload(hc_matrix, img_noisy)
    model_test.apply_denoising()
    
    result = model_test.simulate(shots=shots, monte_carlo_seed=42)
    img_test = result.reconstruct(use_denoising_bias=True)
    metrics = result.compute_metrics(reference_image=img)
    
    psnr = metrics.get('psnr', 0)
    psnr_values.append(psnr)
    print(f"  {shots:5d} shots → PSNR = {psnr:.2f} dB")

# Plot convergence
plt.figure(figsize=(10, 6))
plt.plot(shot_counts, psnr_values, 'o-', linewidth=2, markersize=8)
plt.xlabel('Number of Monte Carlo Shots', fontsize=12)
plt.ylabel('PSNR (dB)', fontsize=12)
plt.title('MHRQI Convergence: Shot Count vs. Reconstruction Quality', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.tight_layout()
plt.show()

## 8. Statistical Analysis

In [ ]:
from scipy import stats

# Run multiple simulations with different seeds
print("Running 5 independent simulations for statistical analysis...\n")
multiple_results = []

for run in range(5):
    model_multi = MHRQI(depth=depth, bit_depth=8)
    model_multi.lazy_upload(hc_matrix, img_noisy)
    model_multi.apply_denoising()
    
    result = model_multi.simulate(shots=5000, monte_carlo_seed=run)
    img_result = result.reconstruct(use_denoising_bias=True)
    metrics = result.compute_metrics(reference_image=img)
    
    multiple_results.append(metrics)
    print(f"Run {run+1}: PSNR = {metrics.get('psnr', 0):.2f} dB, SSIM = {metrics.get('ssim', 0):.4f}")

# Compute statistics
psnr_list = [r.get('psnr', 0) for r in multiple_results]
ssim_list = [r.get('ssim', 0) for r in multiple_results]

print(f"\n=== Statistical Summary ===")
print(f"PSNR: {np.mean(psnr_list):.2f} ± {np.std(psnr_list):.2f} dB")
print(f"SSIM: {np.mean(ssim_list):.4f} ± {np.std(ssim_list):.4f}")

## 9. Summary: Key Takeaways

✅ **Monte Carlo Advantages:**
- Scales to larger images (512×512, 1024×1024) without full 2^n state storage
- GPU acceleration via cuStateVec for efficient sampling
- Configurable accuracy-speed trade-off via shot count
- Realistic noise characterization through statistical sampling

✅ **MHRQI Quantum Denoising:**
- Hierarchical consistency checks preserve anatomical structure
- Native quantum circuit ↔ No post-processing heuristics
- Proven edge preservation (Rank #1 EPF metric)
- Trade-off: structural fidelity vs. absolute speckle suppression

✅ **Medical Imaging Application:**
- Optimized for OCT B-scans with specialized metrics (SSI, ENL, CNR)
- Evaluated on Kermany2018 dataset (32 images, 4 pathologies)
- Outperforms classical methods in edge preservation

---

## Learn More

- 📖 [User Guide](https://keno-00.github.io/MHRQI/guide/)
- 🔧 [API Reference](https://keno-00.github.io/MHRQI/api/)
- 📚 [Research Paper](https://github.com/Keno-00/MHRQI)
- 💻 [GitHub Repository](https://github.com/Keno-00/MHRQI)